# Chạy thử đầu-cuối T3 → T8 trên Kaggle (kho 10k URL đã crawl)

**Cài đặt notebook:** Accelerator = **GPU T4 ×2**, Internet = **On**, Persistence = tuỳ ý.
**Đầu vào (Add Input → Your Datasets):** `mir-code` (thư mục `mir/`, `configs/`) và `mir-data`
(`Data/`, `corpus_clean/`, `chunks/structure/`, `sample/`). Notebook tự tìm đường dẫn, không cần sửa.

Các bước: (1) chuẩn bị → (2) đối chiếu với FlagEmbedding → (3) đo tốc độ → (4) T3 mã hoá → (5) T4 chỉ mục
→ (6) T5 truy hồi + gộp → (7) T6 xếp hạng lại → (8) T8 bài nộp → (9) tổng kết thời gian.
Kho chỉ có ~9 nghìn bài (~0,2% kho thật) nên điểm nếu nộp sẽ rất thấp — mục đích là kiểm tra đường ống và đo thời gian.

In [ ]:
# (1) Chuẩn bị
import os, glob, shutil, subprocess, sys, json, time
def find(marker):
    hits = [p for p in glob.glob(f'/kaggle/input/**/{marker}', recursive=True)]
    assert hits, f'không tìm thấy {marker} trong /kaggle/input'
    return hits[0][:-len(marker) - 1]
CODE_IN, DATA_IN = find('mir/__init__.py'), find('Data/query.parquet')
WORK, CODE = '/kaggle/working', '/kaggle/working/code'
print('code:', CODE_IN, '| data:', DATA_IN)
if not os.path.exists(CODE):
    shutil.copytree(CODE_IN, CODE)
os.chdir(CODE)
os.environ.update(MIR_DATA_DIR=DATA_IN, MIR_WORK_DIR=WORK, HF_HUB_DISABLE_SYMLINKS_WARNING='1')
for name in ('corpus_clean', 'chunks'):
    if not os.path.exists(f'{WORK}/{name}'):
        os.symlink(f'{DATA_IN}/{name}', f'{WORK}/{name}')
# kho nhỏ: khối mã hoá 10.000 đoạn để 2 GPU chia được việc (toàn kho dùng 100.000)
cfg = open('configs/baseline.yaml', encoding='utf-8').read().replace('shard_size: 100000', 'shard_size: 10000')
open('configs/baseline.yaml', 'w', encoding='utf-8').write(cfg)
CFG, T = 'configs/baseline.yaml', {}
def sh(cmd, step=None):
    print('$', cmd, flush=True); t0 = time.time()
    r = subprocess.run(cmd, shell=True)
    if step: T[step] = T.get(step, 0) + time.time() - t0
    assert r.returncode == 0, f'LỖI: {cmd}'
def par(cmds, step):            # chạy song song (mỗi GPU một lệnh), in log
    t0 = time.time()
    ps = [subprocess.Popen(f'{c} > {WORK}/log_{step}_{i}.txt 2>&1', shell=True) for i, c in enumerate(cmds)]
    codes = [p.wait() for p in ps]
    T[step] = time.time() - t0
    for i in range(len(cmds)):
        print(f'--- {step} #{i}\n' + open(f'{WORK}/log_{step}_{i}.txt').read()[-1500:])
    assert all(c == 0 for c in codes), f'LỖI ở {step}'
sh('pip install -q pyvi jieba faiss-cpu')
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU:', torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# (2) Đối chiếu mir.encode với FlagEmbedding (thư viện chính thức) trên 32 đoạn
sh('pip install -q --no-deps --target /kaggle/tmp/flag FlagEmbedding')
sys.path.insert(0, '/kaggle/tmp/flag')
import numpy as np, pyarrow.parquet as pq
from mir import encode
ch = pq.read_table(f'{DATA_IN}/chunks/structure').to_pydict()
titles = encode.load_titles(f'{DATA_IN}/corpus_clean')
texts = encode.make_inputs({'doc_id': ch['doc_id'][:32], 'text': ch['text'][:32]}, titles)
mine = encode.M3Encoder.load('BAAI/bge-m3', 'cuda:0', fp16=True, max_length=512)
d1, s1, _ = mine.encode(texts)
try:
    from FlagEmbedding import BGEM3FlagModel
    ref = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, devices=['cuda:0'])
    out = ref.encode(texts, max_length=512, return_dense=True, return_sparse=True)
    cos = (d1.astype(np.float32) * out['dense_vecs'].astype(np.float32)).sum(1)
    same = sum(set(i.tolist()) == {int(k) for k in lw} for (i, _), lw in zip(s1, out['lexical_weights']))
    worst = max(abs(float(lw.get(str(t), lw.get(int(t), 0))) - float(x)) for (ids, w), lw in zip(s1, out['lexical_weights'])
                for t, x in zip(ids, w))
    print(f'dense cos(mir, Flag) nhỏ nhất = {cos.min():.5f} | sparse: cùng tập token {same}/32 | chênh trọng số lớn nhất = {worst:.4f}')
    del ref
except Exception as e:
    print('Không đối chiếu được với FlagEmbedding:', type(e).__name__, str(e)[:300])
del mine; torch.cuda.empty_cache()

In [ ]:
# (3) Đo tốc độ mã hoá trên 2 GPU cùng lúc (2.000 đoạn mỗi GPU) → ước lượng cho toàn kho
EST = 8_500_000
cmd = (f'python -m mir.encode --config {CFG} --bench 2000 --input {DATA_IN}/chunks/structure '
       f'--clean {DATA_IN}/corpus_clean --est-chunks {EST}')
par([f'{cmd} --device cuda:{g}' for g in range(2)], 'bench')
res = [json.load(open(f'{WORK}/results/t3_bench_cuda{g}.json')) for g in range(2)]
rate = sum(r['chunks_per_s'] for r in res)
print(f'TỔNG 2 GPU: {rate:.0f} đoạn/s | toàn kho {EST:,} đoạn ≈ {EST / rate / 3600:.1f} giờ trên 1 máy T4×2 '
      f'(≈ {EST / rate / 3600 / 3:.1f} giờ/người nếu 3 người) | dung lượng ≈ {res[0]["est_storage_gb"]} GB | '
      f'bộ nhớ GPU tối đa {max(r["peak_gpu_mem_gb"] for r in res)} GB')

In [ ]:
# (4) T3 — mã hoá toàn bộ đoạn (2 GPU, mỗi GPU một nửa số khối)
cmd = f'python -m mir.encode --config {CFG}'
par([f'{cmd} --shard {g}/2 --device cuda:{g}' for g in range(2)], 'T3 mã hoá')
sh(f'du -sh {WORK}/emb/structure; ls {WORK}/emb/structure')

In [ ]:
# (5) T4 — chỉ mục: BM25 (CPU) + dense (FAISS; kho nhỏ → IndexFlatIP). LSR đọc thẳng từ emb ở bước sau.
sh(f'python -m mir.index bm25-tokenize --config {CFG}', 'T4 BM25 tách từ')
sh(f'python -m mir.index bm25-build --config {CFG}', 'T4 BM25 dựng')
sh(f'python -m mir.index dense --config {CFG}', 'T4 dense')
print(open(f'{WORK}/index/structure/dense.json').read())

In [ ]:
# (6) T5 — mã hoá 1.200 câu hỏi, truy hồi 3 nhánh (độ sâu 1000), gộp RRF A–G + có trọng số
sh(f'python -m mir.retrieve qencode --config {CFG} --device cuda:0', 'T5 mã hoá câu hỏi')
sh(f'python -m mir.index dense-tune --config {CFG} --qemb {WORK}/qemb/query.npz', 'T4 đo nlist/nprobe')
sh(f'python -m mir.retrieve run --config {CFG} --branch all', 'T5 truy hồi')
sh(f'python -m mir.fuse --config {CFG}', 'T5 gộp')
import pandas as pd
for f in sorted(glob.glob(f'{WORK}/runs/*.parquet')):
    d = pd.read_parquet(f)
    print(f'{os.path.basename(f):38s} {len(d):>9,} dòng | {d.qid.nunique():>5} câu hỏi có kết quả')

In [ ]:
# (7) T6 — xếp hạng lại top-100 của cấu hình G (Dense+LSR+BM25) bằng ColBERT, 2 GPU
RUN = 'structure_G'
cmd = f'python -m mir.rerank --config {CFG} --run {WORK}/runs/{RUN}.parquet'
par([f'{cmd} --shard {g}/2 --device cuda:{g}' for g in range(2)], 'T6 xếp hạng lại')
sh(f'{cmd} --shard 0/2 --device cuda:0')        # gộp 2 phần → runs/{RUN}_rr.parquet

In [ ]:
# (8) T8 — bài nộp (kiểm tra đủ 1.200 câu, doc_id hợp lệ, chunk_text nguyên văn) → submissions/*.zip
for r in ('structure_G', 'structure_G_rr'):
    sh(f'python -m mir.submit --config {CFG} --run {WORK}/runs/{r}.parquet --name {r} --force', f'T8 {r}')
sub = json.load(open(f'{WORK}/submissions/structure_G_rr.json', encoding='utf-8'))
print(len(sub), 'câu hỏi; ví dụ:', json.dumps(sub[0], ensure_ascii=False)[:600])

In [ ]:
# (9) Tổng kết thời gian và dung lượng → gửi bảng này (cùng kết quả ô 2 và 3) cho người duyệt
n_chunks = pq.read_table(f'{DATA_IN}/chunks/structure', columns=['chunk_id']).num_rows
print(f'Kho thử: {n_chunks:,} đoạn, 1.200 câu hỏi')
for k, v in T.items():
    print(f'  {k:28s} {v / 60:7.1f} phút')
sh(f'du -sh {WORK}/emb {WORK}/index {WORK}/runs {WORK}/submissions; df -h {WORK}')
json.dump({'n_chunks': n_chunks, 'minutes': {k: round(v / 60, 2) for k, v in T.items()}},
          open(f'{WORK}/results/e2e_timing.json', 'w'), indent=1)